# P1-E06 Automated Quantitative Signal-QA and Preprocessing-Path Lock

**Paper 1, Experiment 06 recording-level successor to the passed P1-E05. channel lock**

## Goal

This notebook performs a fully automatic, reproducible signal-quality pilot and locks the preprocessing path without requiring an EEG specialist to complete a review form. It:

1. authenticates the three locked P1-E05.1 inputs;
2. retains recording-level identity and the official CAUEEG splits;
3. selects a duration-stratified pilot from the **training split only**, without diagnosis labels;
4. uses the locked 19-channel selection map instead of resolving channels again;
5. applies identical locked intervals to a minimal reference path and a conservative enhanced path;
6. measures epoch retention, amplitude contamination, flatness, temporal stability, spectral preservation and high-frequency burden automatically;
7. applies prespecified safety gates and locks one preprocessing path before model development; and
8. produces a P1-E07 hand-off for full-cohort preprocessing and automated QC.

Specialist review is **not required** and cannot alter the automatic decision. Independent expert review is retained only as a future validation recommendation.

Run all cells in order. Continue only if the final status is:

> **P1-E06. PASSED — AUTOMATED QA PREPROCESSING PATH LOCKED**

In [ ]:
from pathlib import Path
import hashlib, json, platform, sys, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal

try:
    import mne
    from mne.preprocessing import ICA
except Exception as exc:
    raise ImportError("P1-E06.1 requires MNE. Install with: conda install -c conda-forge mne -y") from exc

PAPER_ROOT=Path(r"D:\EEG_AD_Research\paper1_experiments\paper1_outputs")
E05_ROOT=PAPER_ROOT/'p1_e05_1_channel_montage_lock'
OUTPUT_ROOT=PAPER_ROOT/'p1_e06_1_automated_qa_preprocessing_pilot'
TABLES=OUTPUT_ROOT/'tables'; FIGURES=OUTPUT_ROOT/'figures'; ARRAYS=OUTPUT_ROOT/'pilot_arrays'
OPTIONAL_PANELS=FIGURES/'optional_expert_validation_panels'
for path in [TABLES,FIGURES,ARRAYS,OPTIONAL_PANELS]: path.mkdir(parents=True,exist_ok=True)

RECORDING_MANIFEST=E05_ROOT/'tables'/'13_LOCKED_CHANNEL_HARMONISED_RECORDING_MANIFEST.csv'
INTERVAL_MANIFEST=E05_ROOT/'tables'/'14_LOCKED_CHANNEL_HARMONISED_INTERVAL_MANIFEST.csv'
CHANNEL_MAP=E05_ROOT/'tables'/'12_LOCKED_CANONICAL_19_EEG_CHANNEL_SELECTION_MAP.csv'
E05_RUN_RECORD=E05_ROOT/'run_record.json'

SEED=20260828; N_PILOT=24; TARGET_EPOCHS=12; MIN_EPOCHS=6
FS_TARGET=200.0; BAND=(1.0,40.0); EPOCH_SECONDS=10.0; EDGE_TRIM_SECONDS=2.0
MAX_ICA_COMPONENTS=2; MAX_REPAIRED_CHANNELS=2
CANONICAL=['Fp1','Fp2','F7','F3','Fz','F4','F8','T3','C3','Cz','C4','T4','T5','P3','Pz','P4','T6','O1','O2']

EQUIVALENCE_MARGIN=0.02
BOOTSTRAP_REPS=5000
QA_WEIGHTS={'retention':0.25,'epoch_acceptance':0.20,'clean_signal':0.15,
            'temporal_stability':0.15,'spectral_preservation':0.15,'low_hf_burden':0.10}

plt.rcParams.update({'figure.dpi':120,'savefig.dpi':600,'font.size':10,'axes.titlesize':11,
                     'axes.labelsize':10,'pdf.fonttype':42,'ps.fonttype':42,'savefig.bbox':'tight'})
sns.set_theme(style='whitegrid',context='paper'); mne.set_log_level('ERROR')

def sha256_file(path,block_size=4*1024*1024):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(block_size),b''): digest.update(block)
    return digest.hexdigest()
def save_table(frame,name):
    frame.to_csv(TABLES/f'{name}.csv',index=False)
    try: frame.to_excel(TABLES/f'{name}.xlsx',index=False)
    except Exception as exc: warnings.warn(f'Excel export skipped ({exc}); CSV was saved.')
def savefig(fig,name):
    fig.savefig(FIGURES/f'{name}.png',dpi=600,bbox_inches='tight')
    fig.savefig(FIGURES/f'{name}.pdf',bbox_inches='tight')
def stable_u(value):
    return int(hashlib.sha256(f'{SEED}|{value}'.encode()).hexdigest()[:16],16)/16**16

print('Experiment: P1-E06.1')
print('Python',sys.version.split()[0],'MNE',mne.__version__)
print('Input:',E05_ROOT)
print('Output:',OUTPUT_ROOT)

## 1. Authenticate the P1-E05.1 channel and montage lock

The recording, interval and channel-selection manifests must reproduce the hashes in the passed P1-E05.1 run record. No older P1-E04 or P1-E05 file is accepted directly.

In [ ]:
required=[RECORDING_MANIFEST,INTERVAL_MANIFEST,CHANNEL_MAP,E05_RUN_RECORD]
missing=[str(path) for path in required if not path.is_file()]
if missing: raise FileNotFoundError('Run and pass P1-E05.1 first. Missing:\n'+'\n'.join(missing))

e05_run=json.loads(E05_RUN_RECORD.read_text(encoding='utf-8'))
expected={
    'recordings':e05_run.get('locked_recording_manifest_sha256',''),
    'intervals':e05_run.get('locked_interval_manifest_sha256',''),
    'channel_map':e05_run.get('locked_channel_map_sha256','')}
observed={'recordings':sha256_file(RECORDING_MANIFEST),
          'intervals':sha256_file(INTERVAL_MANIFEST),'channel_map':sha256_file(CHANNEL_MAP)}

recordings=pd.read_csv(RECORDING_MANIFEST,dtype={'recording_id':str,'serial':str})
intervals=pd.read_csv(INTERVAL_MANIFEST,dtype={'recording_id':str,'serial':str})
channel_map=pd.read_csv(CHANNEL_MAP,dtype={'recording_id':str,'serial':str})
for frame in (recordings,intervals,channel_map):
    for column in ('recording_id','serial'):
        if column in frame: frame[column]=frame[column].astype(str).str.strip().str.zfill(5)

required_recording={'recording_id','serial','edf_path','official_split','experimental_label',
                    'analysis_unit','participant_identity_verified','retained_seconds'}
required_interval={'recording_id','interval_number','start_seconds','stop_seconds','duration_seconds'}
required_map={'recording_id','original_index','original_name','canonical_index','canonical_name'}
for name,frame,columns in [('recording',recordings,required_recording),
                           ('interval',intervals,required_interval),('channel map',channel_map,required_map)]:
    absent=columns-set(frame.columns)
    if absent: raise ValueError(f'{name} manifest missing columns: {sorted(absent)}')

map_counts=channel_map.groupby('recording_id').size()
map_order=channel_map.sort_values(['recording_id','canonical_index']).groupby('recording_id').canonical_name.apply(list)
checks=pd.DataFrame([
 {'check':'P1-E05.1 passed','pass':str(e05_run.get('status','')).startswith('P1-E05.1 PASSED'),'detail':e05_run.get('status','')},
 {'check':'all three SHA-256 values authenticated','pass':all(expected[k] and expected[k]==observed[k] for k in expected),'detail':observed},
 {'check':'979 recording-level inputs','pass':len(recordings)==979 and recordings.recording_id.nunique()==979,'detail':len(recordings)},
 {'check':'8,557 locked intervals','pass':len(intervals)==8557,'detail':len(intervals)},
 {'check':'19 map rows per recording','pass':len(map_counts)==979 and map_counts.eq(19).all(),'detail':map_counts.value_counts().to_dict()},
 {'check':'canonical order reproduced','pass':len(map_order)==979 and map_order.map(lambda x:x==CANONICAL).all(),'detail':'19-channel fixed order'},
 {'check':'official splits complete','pass':set(recordings.official_split)=={'train','validation','test'},'detail':recordings.official_split.value_counts().to_dict()},
 {'check':'analysis remains recording-level','pass':recordings.analysis_unit.astype(str).str.lower().eq('recording').all(),'detail':recordings.analysis_unit.value_counts().to_dict()},
 {'check':'participant identity remains unverified','pass':~recordings.participant_identity_verified.astype(str).str.lower().isin(['true','1']).any(),'detail':recordings.participant_identity_verified.value_counts(dropna=False).to_dict()},
])
save_table(checks,'00_p1_e05_1_input_authentication_checks'); display(checks)
if not checks['pass'].all(): raise AssertionError('STOP — P1-E05.1 INPUT AUTHENTICATION FAILED')
print('P1-E05.1 INPUTS AUTHENTICATED')

## 2. Apply the duration gate and select a training-only automatic-QA pilot

The pilot is selected only from the official training split. Diagnosis is not used. Recordings are stratified by retained duration and selected deterministically. Validation and test recordings remain untouched while the preprocessing path is chosen.

In [ ]:
capacity=(intervals.assign(
    usable_seconds=lambda d:(d.duration_seconds-2*EDGE_TRIM_SECONDS).clip(lower=0),
    theoretical_epochs=lambda d:np.floor(d.usable_seconds/EPOCH_SECONDS).astype(int))
    .groupby('recording_id',as_index=False).theoretical_epochs.sum())

# Only split membership and duration are exposed during pilot selection.
pilot_frame=recordings[['recording_id','serial','edf_path','official_split','retained_seconds']].copy()
pilot_frame=pilot_frame.merge(capacity,on='recording_id',how='left',validate='one_to_one')
pilot_frame['theoretical_epochs']=pilot_frame.theoretical_epochs.fillna(0).astype(int)
pilot_frame['duration_feasible']=pilot_frame.theoretical_epochs.ge(TARGET_EPOCHS)
save_table(pilot_frame[['recording_id','serial','official_split','retained_seconds',
                        'theoretical_epochs','duration_feasible']],
           '01_preprocessing_duration_feasibility')

pilot_pool=pilot_frame.loc[pilot_frame.official_split.eq('train') & pilot_frame.duration_feasible].copy()
if len(pilot_pool)<N_PILOT: raise ValueError(f'Only {len(pilot_pool)} training recordings can supply the target epochs')
pilot_pool['duration_stratum']=pd.qcut(pilot_pool.retained_seconds.rank(method='first'),4,
                                      labels=['Q1','Q2','Q3','Q4'])
pilot_pool['_u']=pilot_pool.recording_id.map(stable_u)
per=N_PILOT//4
pilot=pd.concat([group.sort_values('_u').head(per)
                 for _,group in pilot_pool.groupby('duration_stratum',observed=True)])
pilot=pilot.sort_values(['duration_stratum','_u']).head(N_PILOT).copy()
pilot['case_id']=[f'CASE_{i:03d}' for i in range(1,len(pilot)+1)]
pilot_intervals=intervals.loc[intervals.recording_id.isin(pilot.recording_id)].copy()
if len(pilot)!=N_PILOT or pilot.theoretical_epochs.min()<TARGET_EPOCHS:
    raise AssertionError('Automatic-QA pilot construction failed')

# Identity key is retained for provenance; it is not needed by any human reviewer.
pilot[['case_id','recording_id','serial','edf_path','official_split','duration_stratum',
       'retained_seconds','theoretical_epochs']].to_csv(
    TABLES/'02_PILOT_RECORDING_PROVENANCE_KEY.csv',index=False)
save_table(pilot[['case_id','duration_stratum','retained_seconds','theoretical_epochs']],
           '03_automatic_qa_pilot_case_manifest')
fig,ax=plt.subplots(figsize=(6.2,3.8))
sns.boxplot(data=pilot,x='duration_stratum',y='retained_seconds',ax=ax,color='#8ecae6')
sns.stripplot(data=pilot,x='duration_stratum',y='retained_seconds',ax=ax,color='black',size=3)
ax.set(title='Training-only automated-QA pilot',xlabel='Retained-duration stratum',
       ylabel='Eligible retained seconds')
savefig(fig,'Figure_01_training_only_pilot_duration'); plt.show()
display(pilot[['case_id','duration_stratum','retained_seconds','theoretical_epochs']])

## 3. Signal-processing and automatic-quality functions

Both paths use the original CAUEEG average-referenced EEG channels in the exact P1-E05.1 canonical order. The notebook does **not** apply a second common-average reference. Each retained interval is detrended and zero-phase band-pass filtered from 1–40 Hz. The enhanced path additionally permits conservative ICA and repair of at most two focal channel outliers.

Automated metrics—not human ratings—determine epoch acceptance and path selection. Specialist evaluation is recommended only as future independent validation.

In [ ]:
def read_locked_channels(path,recording_id):
    raw=mne.io.read_raw_edf(str(path),preload=False,verbose='ERROR')
    rows=(channel_map.loc[channel_map.recording_id.eq(recording_id)]
          .sort_values('canonical_index'))
    if len(rows)!=len(CANONICAL) or rows.canonical_name.tolist()!=CANONICAL:
        raw.close(); raise ValueError(f'Locked channel map invalid for recording {recording_id}')
    picks=rows.original_index.astype(int).tolist()
    if max(picks)>=len(raw.ch_names):
        raw.close(); raise ValueError(f'Locked channel index exceeds EDF header for {recording_id}')
    observed=[raw.ch_names[index] for index in picks]
    if observed!=rows.original_name.tolist():
        raw.close(); raise ValueError(f'EDF channel labels changed after P1-E05.1 for {recording_id}')
    return raw,picks

def extract_intervals(raw,picks,rows):
    fs=float(raw.info['sfreq']); chunks=[]
    for row in rows.itertuples():
        start=max(0,int(round(float(row.start_seconds)*fs)))
        stop=min(raw.n_times,int(round(float(row.stop_seconds)*fs)))
        if stop-start>=fs*(EPOCH_SECONDS+2*EDGE_TRIM_SECONDS):
            chunks.append(raw.get_data(picks=picks,start=start,stop=stop)*1e6)
    if not chunks: raise ValueError('No locked interval long enough for preprocessing')
    return chunks,fs

def base_process(x,fs):
    x=signal.detrend(x,axis=1,type='linear')
    sos=signal.butter(4,[BAND[0],BAND[1]],btype='bandpass',fs=fs,output='sos')
    x=signal.sosfiltfilt(sos,x,axis=1)
    if fs!=FS_TARGET: x=signal.resample_poly(x,int(FS_TARGET),int(round(fs)),axis=1)
    return x
def rz(v):
    v=np.asarray(v,float); med=np.nanmedian(v); mad=np.nanmedian(np.abs(v-med)); return (v-med)/(1.4826*mad+1e-12)
def channel_scores(x,fs):
    sd=np.std(x,axis=1); dx=np.std(np.diff(x,axis=1),axis=1); line=np.mean(np.abs(x)>150,axis=1)
    f,p=signal.welch(x,fs=fs,nperseg=min(x.shape[1],int(4*fs)),axis=1)
    hf=np.trapezoid(p[:,(f>=30)&(f<=40)],f[(f>=30)&(f<=40)],axis=1)/(np.trapezoid(p[:,(f>=1)&(f<=40)],f[(f>=1)&(f<=40)],axis=1)+1e-12)
    return np.maximum.reduce([np.abs(rz(np.log(sd+1e-9))),np.abs(rz(np.log(dx+1e-9))),np.abs(rz(hf+1e-9)),np.abs(rz(line+1e-9))])
def enhanced_process(chunks,fs):
    base=[base_process(x,fs) for x in chunks]; cat=np.concatenate(base,axis=1)
    scores=channel_scores(cat,FS_TARGET); bad=np.where(scores>5.0)[0].tolist()[:MAX_REPAIRED_CHANNELS]
    # MNE montage lookup is case-sensitive. Keep the published legacy labels in
    # saved arrays, but use standard_1020 names internally for interpolation.
    mne_name={'FP1':'Fp1','FP2':'Fp2','FZ':'Fz','CZ':'Cz','PZ':'Pz',
              'T3':'T7','T4':'T8','T5':'P7','T6':'P8'}
    modern=[mne_name.get(c,c) for c in CANONICAL]
    info=mne.create_info(modern,FS_TARGET,'eeg'); info.set_montage('standard_1020',on_missing='raise')
    fit=mne.io.RawArray(cat[:,:min(cat.shape[1],int(180*FS_TARGET))]*1e-6,info,verbose='ERROR')
    ncomp=min(15,len(CANONICAL)-1); ica=ICA(n_components=ncomp,method='fastica',random_state=SEED,max_iter=500)
    exclude=[]
    try:
        ica.fit(fit,verbose='ERROR'); src=ica.get_sources(fit).get_data(); frontal=np.mean(fit.get_data(picks=[0,1,2,6]),axis=0)
        cor=np.array([abs(np.corrcoef(s,frontal)[0,1]) for s in src]); kurt=np.array([np.mean(((s-s.mean())/(s.std()+1e-12))**4) for s in src])
        rank=np.argsort(np.maximum(cor/0.45,kurt/12))[::-1]
        exclude=[int(i) for i in rank if cor[i]>.45 or kurt[i]>12][:MAX_ICA_COMPONENTS]
    except Exception as e: warnings.warn(f'ICA skipped for this participant: {e}'); ica=None
    out=[]
    for x in base:
        rr=mne.io.RawArray(x*1e-6,info,verbose='ERROR')
        if ica is not None and exclude: ica.apply(rr,exclude=exclude,verbose='ERROR')
        if bad:
            rr.info['bads']=[modern[i] for i in bad]
            try:
                rr.interpolate_bads(reset_bads=True,mode='accurate',verbose='ERROR')
            except Exception as interpolation_error:
                # A deterministic robust fallback prevents a montage-specific
                # library failure from erasing the entire pilot. It replaces
                # only the predeclared focal outliers and is explicitly logged.
                warnings.warn(f'Spherical interpolation failed; using channel-median fallback: {interpolation_error}')
                data=rr.get_data(); good=[i for i in range(len(modern)) if i not in bad]
                replacement=np.median(data[good],axis=0)
                for i in bad: data[i]=replacement
                rr=mne.io.RawArray(data,info,verbose='ERROR'); rr.info['bads']=[]
        out.append(rr.get_data()*1e6)
    return out,bad,exclude,scores
def make_epochs(chunks):
    L=int(EPOCH_SECONDS*FS_TARGET); trim=int(EDGE_TRIM_SECONDS*FS_TARGET); out=[]
    for x in chunks:
        x=x[:,trim:x.shape[1]-trim] if x.shape[1]>2*trim else x
        for a in range(0,x.shape[1]-L+1,L): out.append(x[:,a:a+L])
    return np.asarray(out,float)
def epoch_metrics(ep):
    ptp=np.ptp(ep,axis=2).max(axis=1); rms=np.sqrt(np.mean(ep**2,axis=(1,2))); step=np.max(np.abs(np.diff(ep,axis=2)),axis=(1,2)); clip=np.mean(np.abs(ep)>=150,axis=(1,2))
    return pd.DataFrame({'ptp_uv':ptp,'rms_uv':rms,'max_step_uv':step,'clip_fraction':clip})
def accept_epochs(m):
    score=np.maximum.reduce([np.abs(rz(np.log(m.ptp_uv+1e-9))),np.abs(rz(np.log(m.rms_uv+1e-9))),np.abs(rz(np.log(m.max_step_uv+1e-9))),np.abs(rz(m.clip_fraction+1e-9))])
    return (score<5)&(m.ptp_uv<600)&(m.rms_uv<120)&(m.clip_fraction<.02),score

## 4. Run both paths on identical locked pilot intervals

This is the computationally expensive cell. It saves compact arrays, automatic QC metrics and explicit processing failures. Every pilot recording must complete both paths before the decision stage can run.

In [ ]:
records=[]; failures=[]; arrays={}
for n,row in enumerate(pilot.itertuples(),1):
    try:
        raw,picks=read_locked_channels(row.edf_path,row.recording_id); ir=pilot_intervals[pilot_intervals.recording_id==row.recording_id].sort_values('start_seconds')
        chunks,fs=extract_intervals(raw,picks,ir); ref=[base_process(x,fs) for x in chunks]
        enh,bad,excluded,chscore=enhanced_process(chunks,fs)
        for path_name,processed in [('reference',ref),('enhanced',enh)]:
            ep=make_epochs(processed); met=epoch_metrics(ep); ok,q=accept_epochs(met); keep=ep[ok][:TARGET_EPOCHS]
            if len(keep)<MIN_EPOCHS: raise ValueError(f'{path_name}: only {len(keep)} accepted epochs')
            arrays[(row.case_id,path_name)]=keep.astype('float32')
            rr={'case_id':row.case_id,'path':path_name,'candidate_epochs':len(ep),'accepted_epochs':int(ok.sum()),'retained_epochs':len(keep),'accepted_fraction':float(ok.mean()),
                'median_ptp_uv':float(met.loc[ok,'ptp_uv'].median()),'median_rms_uv':float(met.loc[ok,'rms_uv'].median()),'median_clip_fraction':float(met.loc[ok,'clip_fraction'].median()),
                'repaired_channels':';'.join(CANONICAL[i] for i in bad) if path_name=='enhanced' else '', 'n_repaired':len(bad) if path_name=='enhanced' else 0,
                'ica_components':';'.join(map(str,excluded)) if path_name=='enhanced' else '', 'n_ica_removed':len(excluded) if path_name=='enhanced' else 0}
            records.append(rr)
        print(f'[{n:02d}/{len(pilot)}] {row.case_id} complete')
    except Exception as e:
        failures.append({'case_id':row.case_id,'error_type':type(e).__name__,'error':str(e)}); print(row.case_id,'FAILED:',e)
qc_columns=['case_id','path','candidate_epochs','accepted_epochs','retained_epochs','accepted_fraction',
            'median_ptp_uv','median_rms_uv','median_clip_fraction','repaired_channels','n_repaired',
            'ica_components','n_ica_removed']
failure_columns=['case_id','error_type','error']
qc=pd.DataFrame(records,columns=qc_columns); fail=pd.DataFrame(failures,columns=failure_columns)
save_table(qc,'04_path_epoch_quality'); save_table(fail,'05_processing_failures')
for (case,path_name),ep in arrays.items(): np.savez_compressed(ARRAYS/f'{case}_{path_name}.npz',epochs_uv=ep,fs=np.float32(FS_TARGET),channels=np.asarray(CANONICAL,dtype='U8'))
if qc.empty:
    print('All pilot cases failed before a path-level QC row could be created.')
    display(fail.head(24))
    first='; '.join(f"{r.case_id}: {r.error_type}: {r.error}" for r in fail.head(5).itertuples())
    raise RuntimeError('P1-E05 processing produced zero successful cases. Original errors: '+first+
                       f'. Full errors saved to {TABLES / "05_processing_failures.csv"}')
ready=set(qc.groupby('case_id').filter(lambda g:set(g.path)=={'reference','enhanced'}).case_id)
if len(ready)!=len(pilot):
    print(f'Only {len(ready)}/{len(pilot)} cases completed both paths. Original failures:')
    display(fail.head(24))
    raise RuntimeError(f'Pilot incomplete; inspect {TABLES / "05_processing_failures.csv"}. Do not continue.')
display(qc.groupby('path').agg(participants=('case_id','nunique'),median_retained=('retained_epochs','median'),median_acceptance=('accepted_fraction','median'),median_ptp=('median_ptp_uv','median')))
fig,axes=plt.subplots(1,2,figsize=(8.5,3.6))
sns.boxplot(data=qc,x='path',y='accepted_fraction',ax=axes[0],color='#90be6d'); sns.stripplot(data=qc,x='path',y='accepted_fraction',ax=axes[0],color='black',size=2)
sns.boxplot(data=qc,x='path',y='median_ptp_uv',ax=axes[1],color='#f9c74f'); sns.stripplot(data=qc,x='path',y='median_ptp_uv',ax=axes[1],color='black',size=2)
axes[0].set_title('Accepted epoch fraction'); axes[1].set_title('Median peak-to-peak amplitude'); savefig(fig,'Figure_02_objective_path_quality'); plt.show()

## 5. Derive prespecified recording-level automated-QA metrics

For recording (i) and path (p), six utilities are bounded to ([0,1]):

\[
Q_{ip}=0.25R_{ip}+0.20A_{ip}+0.15C_{ip}+0.15S_{ip}+0.15P_{ip}+0.10L_{ip}.
\]

- (R): processing success/retention;
- (A=\min(a/0.80,1)): accepted-epoch fraction utility;
- (C=1-\min(c/0.02,1)): inverse extreme-amplitude/clip burden;
- (S=1-\min(\mathrm{rCV}_{RMS}/0.50,1)): temporal stability;
- (P=\exp[-\mathrm{median}_{2-30\,Hz}|\log(PSD_p/PSD_{reference})|]): preservation of the minimally processed physiological spectrum;
- (L=1-\min(h/0.20,1)): inverse 30–40 Hz burden, where (h=P_{30-40}/P_{1-40}).

The reference path has (P=1) by definition. This does not force its selection: enhanced preprocessing can compensate through objectively lower artifact burden and better stability, but it cannot win by destroying physiological spectral structure.

In [ ]:
EPS=1e-12

def robust_cv(values):
    values=np.asarray(values,float)
    med=np.median(values)
    return float(1.4826*np.median(np.abs(values-med))/(abs(med)+EPS))

def array_qa(case_id,path_name,epochs):
    epochs=np.asarray(epochs,float)
    finite=bool(np.isfinite(epochs).all())
    channel_sd=np.std(epochs,axis=(0,2))
    flat_channels=int(np.sum(channel_sd<1e-6))
    clip_fraction=float(np.mean(np.abs(epochs)>=150))
    epoch_rms=np.sqrt(np.mean(epochs**2,axis=(1,2)))
    temporal_rcv=robust_cv(epoch_rms)
    f,pow_=signal.welch(epochs,fs=FS_TARGET,nperseg=int(4*FS_TARGET),axis=-1)
    psd=np.median(pow_,axis=(0,1))
    total=np.trapezoid(psd[(f>=1)&(f<=40)],f[(f>=1)&(f<=40)])+EPS
    hf=float(np.trapezoid(psd[(f>=30)&(f<=40)],f[(f>=30)&(f<=40)])/total)
    return {'case_id':case_id,'path':path_name,'finite':finite,'flat_channels':flat_channels,
            'clip_fraction_array':clip_fraction,'temporal_rms_rcv':temporal_rcv,
            'hf_30_40_fraction':hf,'freqs':f,'psd':psd}

details=[]
for case in pilot.case_id:
    for path_name in ['reference','enhanced']:
        details.append(array_qa(case,path_name,arrays[(case,path_name)]))

# Spectral preservation is paired within participant and calculated over 2–30 Hz.
lookup={(r['case_id'],r['path']):r for r in details}
for case in pilot.case_id:
    ref=lookup[(case,'reference')]; use=(ref['freqs']>=2)&(ref['freqs']<=30)
    ref_psd=ref['psd'][use]
    for path_name in ['reference','enhanced']:
        cur=lookup[(case,path_name)]['psd'][use]
        distortion=float(np.median(np.abs(np.log((cur+EPS)/(ref_psd+EPS)))))
        lookup[(case,path_name)]['spectral_log_distortion']=distortion
        lookup[(case,path_name)]['spectral_preservation']=float(np.exp(-distortion))

metric_rows=[]
for r in details:
    qrow=qc[(qc.case_id==r['case_id'])&(qc.path==r['path'])].iloc[0]
    retention=float(qrow.retained_epochs/TARGET_EPOCHS)
    acceptance=float(qrow.accepted_fraction)
    utilities={
        'retention':np.clip(retention,0,1),
        'epoch_acceptance':np.clip(acceptance/0.80,0,1),
        'clean_signal':np.clip(1-r['clip_fraction_array']/0.02,0,1),
        'temporal_stability':np.clip(1-r['temporal_rms_rcv']/0.50,0,1),
        'spectral_preservation':np.clip(r['spectral_preservation'],0,1),
        'low_hf_burden':np.clip(1-r['hf_30_40_fraction']/0.20,0,1)}
    composite=float(sum(QA_WEIGHTS[k]*utilities[k] for k in QA_WEIGHTS))
    metric_rows.append({'case_id':r['case_id'],'path':r['path'],'processing_success':1,
        'retention_fraction':retention,'accepted_fraction':acceptance,'finite':r['finite'],
        'flat_channels':r['flat_channels'],'clip_fraction':r['clip_fraction_array'],
        'temporal_rms_rcv':r['temporal_rms_rcv'],'spectral_log_distortion':r['spectral_log_distortion'],
        'spectral_preservation':r['spectral_preservation'],'hf_30_40_fraction':r['hf_30_40_fraction'],
        **{f'utility_{k}':float(v) for k,v in utilities.items()},'composite_Q':composite})

recording_qa=pd.DataFrame(metric_rows)
save_table(recording_qa,'06_recording_level_automated_QA')
display(recording_qa.head(8))

## 6. Apply safety gates, quantify uncertainty and lock the path

Safety gates override the composite score. A path must satisfy:

- processing success (≥95\%);
- median accepted-epoch fraction (≥80\%);
- minimum recording accepted fraction (≥50\%);
- no non-finite samples or flat channels;
- median clip fraction (≤1\%\);
- enhanced spectral preservation (≥85\%\);
- enhanced mean acceptance no more than 0.05 below reference.

If both paths pass, the larger mean (Q) is selected. When (|\Delta Q|<0.02), the simpler reference path is selected. The paired recording bootstrap reports uncertainty but does not alter this frozen rule.

In [ ]:
summary=(recording_qa.groupby('path',as_index=False)
    .agg(participants=('case_id','nunique'),processing_success=('processing_success','mean'),
         median_acceptance=('accepted_fraction','median'),minimum_acceptance=('accepted_fraction','min'),
         finite_fraction=('finite','mean'),maximum_flat_channels=('flat_channels','max'),
         median_clip_fraction=('clip_fraction','median'),median_temporal_rcv=('temporal_rms_rcv','median'),
         median_spectral_preservation=('spectral_preservation','median'),median_hf_burden=('hf_30_40_fraction','median'),
         mean_composite_Q=('composite_Q','mean'),sd_composite_Q=('composite_Q','std')))

s=summary.set_index('path')
mean_accept=recording_qa.groupby('path').accepted_fraction.mean()
gates_by_path={}
for path_name in ['reference','enhanced']:
    gates_by_path[path_name]={
        'processing_success_ge_0.95':bool(s.loc[path_name,'processing_success']>=.95),
        'median_acceptance_ge_0.80':bool(s.loc[path_name,'median_acceptance']>=.80),
        'minimum_acceptance_ge_0.50':bool(s.loc[path_name,'minimum_acceptance']>=.50),
        'all_arrays_finite':bool(s.loc[path_name,'finite_fraction']==1),
        'no_flat_channels':bool(s.loc[path_name,'maximum_flat_channels']==0),
        'median_clip_fraction_le_0.01':bool(s.loc[path_name,'median_clip_fraction']<=.01)}
gates_by_path['enhanced']['spectral_preservation_ge_0.85']=bool(s.loc['enhanced','median_spectral_preservation']>=.85)
gates_by_path['enhanced']['acceptance_noninferiority_minus_0.05']=bool(mean_accept['enhanced']>=mean_accept['reference']-.05)

passes={p:all(gates_by_path[p].values()) for p in ['reference','enhanced']}
delta=float(s.loc['enhanced','mean_composite_Q']-s.loc['reference','mean_composite_Q'])
if passes['enhanced'] and not passes['reference']:
    selected='enhanced'; reason='Enhanced was the only path passing all safety gates.'
elif passes['reference'] and not passes['enhanced']:
    selected='reference'; reason='Reference was the only path passing all safety gates.'
elif not any(passes.values()):
    selected=None; reason='Neither path passed all prespecified safety gates.'
elif abs(delta)<EQUIVALENCE_MARGIN:
    selected='reference'; reason=f'Paths were equivalent (|delta Q|={abs(delta):.4f}<0.02); simpler reference path selected.'
else:
    selected='enhanced' if delta>0 else 'reference'; reason='Both paths passed; path with larger prespecified composite score selected.'

# Paired recording bootstrap CI for enhanced minus reference composite score.
wide=recording_qa.pivot(index='case_id',columns='path',values='composite_Q').dropna()
paired=(wide.enhanced-wide.reference).to_numpy()
rng=np.random.default_rng(SEED)
boot=np.mean(rng.choice(paired,size=(BOOTSTRAP_REPS,len(paired)),replace=True),axis=1)
ci_low,ci_high=np.quantile(boot,[.025,.975])
summary['passed_all_safety_gates']=summary.path.map(passes)
save_table(summary,'07_path_level_automated_QA_summary')

decision={'experiment':'P1-E06.1','status':'P1-E06.1 PASSED — AUTOMATED QA PREPROCESSING PATH LOCKED' if selected else 'STOP — NO PATH PASSED SAFETY GATES',
          'selected_primary_path':selected,'decision_reason':reason,'composite_formula':QA_WEIGHTS,
          'equivalence_margin':EQUIVALENCE_MARGIN,'enhanced_minus_reference_mean_Q':delta,
          'paired_bootstrap_95CI':[float(ci_low),float(ci_high)],'bootstrap_repetitions':BOOTSTRAP_REPS,
          'safety_gates':gates_by_path,'path_passes_all_gates':passes,'n_pilot':N_PILOT,'pilot_source_split':'train',
          'diagnostic_labels_used':False,'classifier_performance_used':False,'manual_review_required':False}
(OUTPUT_ROOT/'p1_e06_1_automated_qa_preprocessing_path_lock.json').write_text(json.dumps(decision,indent=2),encoding='utf-8')
save_table(pd.DataFrame([{'path':p,'gate':k,'pass':v} for p,d in gates_by_path.items() for k,v in d.items()]),'08_safety_gate_results')
display(summary); display(pd.DataFrame([decision]).T.rename(columns={0:'value'}))
if selected is None:
    raise RuntimeError(reason+' Review 06_recording_level_automated_QA.csv before proceeding.')

## 7. Save publication-ready automatic-QA figures

All figures describe objective signal-quality metrics only. They contain no diagnostic labels, classifier results or specialist ratings.

In [ ]:
# Figure 1 — recording-level metric comparison.
plot_metrics=['accepted_fraction','clip_fraction','temporal_rms_rcv','spectral_preservation','hf_30_40_fraction','composite_Q']
labels=['Accepted epoch fraction','Extreme-amplitude fraction','Temporal RMS rCV','Spectral preservation','30–40 Hz fraction','Composite quality score']
fig,axes=plt.subplots(2,3,figsize=(11,7))
for ax,m,label in zip(axes.flat,plot_metrics,labels):
    sns.boxplot(data=recording_qa,x='path',y=m,ax=ax,color='#8ecae6',showfliers=False)
    sns.stripplot(data=recording_qa,x='path',y=m,ax=ax,color='#1d3557',size=2.6,alpha=.7)
    ax.set_title(label); ax.set_xlabel(''); ax.set_ylabel('')
fig.suptitle('Prespecified automated preprocessing quality assessment',fontsize=14)
fig.tight_layout(); savefig(fig,'Figure_03_automated_QA_metric_comparison'); plt.show()

# Figure 2 — component utilities and locked decision.
utility_cols=[f'utility_{k}' for k in QA_WEIGHTS]
long=(recording_qa.groupby('path')[utility_cols].mean().reset_index()
      .melt(id_vars='path',var_name='component',value_name='mean_utility'))
long['component']=long.component.str.replace('utility_','',regex=False).str.replace('_',' ')
fig,axes=plt.subplots(1,2,figsize=(11,4.2))
sns.barplot(data=long,x='component',y='mean_utility',hue='path',ax=axes[0],palette=['#457b9d','#e76f51'])
axes[0].set_ylim(0,1.05); axes[0].tick_params(axis='x',rotation=35); axes[0].set_title('Mean component utilities'); axes[0].set_xlabel(''); axes[0].set_ylabel('Utility (0–1)')
sns.barplot(data=summary,x='path',y='mean_composite_Q',ax=axes[1],palette=['#457b9d','#e76f51'],hue='path',legend=False)
axes[1].errorbar(np.arange(len(summary)),summary.mean_composite_Q,yerr=summary.sd_composite_Q,fmt='none',c='black',capsize=4)
axes[1].set_ylim(0,1.05); axes[1].set_title(f'Composite Q; locked path: {selected}'); axes[1].set_xlabel(''); axes[1].set_ylabel('Mean ± SD')
fig.tight_layout(); savefig(fig,'Figure_04_composite_components_and_locked_decision'); plt.show()

# Figure 3 — paired participant differences and bootstrap distribution.
fig,axes=plt.subplots(1,2,figsize=(10,4))
for case,row in wide.iterrows(): axes[0].plot(['reference','enhanced'],[row.reference,row.enhanced],color='0.65',lw=.7,alpha=.7)
axes[0].scatter(np.zeros(len(wide)),wide.reference,s=14,label='Reference'); axes[0].scatter(np.ones(len(wide)),wide.enhanced,s=14,label='Enhanced')
axes[0].set(xlim=(-.25,1.25),xticks=[0,1],xticklabels=['Reference','Enhanced'],ylabel='Composite Q',title='Paired recording scores')
sns.histplot(boot,bins=35,kde=True,ax=axes[1],color='#457b9d'); axes[1].axvline(0,c='black',ls='--'); axes[1].axvspan(ci_low,ci_high,color='#e9c46a',alpha=.35)
axes[1].set_title(f'Paired bootstrap ΔQ\n95% CI [{ci_low:.3f}, {ci_high:.3f}]'); axes[1].set_xlabel('Enhanced − reference')
fig.tight_layout(); savefig(fig,'Figure_05_paired_composite_and_bootstrap'); plt.show()
print('Publication figures saved to:',FIGURES)

## 8. Export optional panels for future expert validation

These waveform and PSD panels are documentary outputs only. They are **not required**, no specialist form is needed, and they cannot override the automatic decision. They may be retained for a later independent validation study if expert access becomes available.

In [ ]:
import shutil
from IPython.display import Image, display
show_idx=[0,1,3,6,7,8,10,17]
for case in pilot.case_id:
    order=['reference','enhanced'] if stable_u(case)<.5 else ['enhanced','reference']
    fig,axes=plt.subplots(2,2,figsize=(13,8.2)); fig.suptitle(f'{case} — optional future expert-validation panel',fontsize=14)
    for col,(lab,path_name) in enumerate(zip(['A','B'],order)):
        ep=arrays[(case,path_name)]; x=ep[0]; t=np.arange(x.shape[1])/FS_TARGET; scale=np.nanmedian(np.std(x,axis=1))*5+EPS
        for j,ch in enumerate(show_idx): axes[0,col].plot(t,x[ch]/scale+j,color=plt.cm.tab10(j%10),lw=.65)
        axes[0,col].set(yticks=range(len(show_idx)),yticklabels=[CANONICAL[j] for j in show_idx],xlabel='Seconds',title=f'{case} — Path {lab}: waveform')
        f,pow_=signal.welch(ep,fs=FS_TARGET,nperseg=int(4*FS_TARGET),axis=-1); psd=np.median(pow_,axis=(0,1)); use=(f>=1)&(f<=50)
        axes[1,col].semilogy(f[use],psd[use],lw=1.2); axes[1,col].axvspan(8,13,color='#fee89a',alpha=.5)
        axes[1,col].set(xlabel='Frequency (Hz)',ylabel='PSD (µV²/Hz)',title=f'{case} — Path {lab}: channel/epoch-median PSD')
    fig.tight_layout(); fig.savefig(OPTIONAL_PANELS/f'{case}.png',dpi=600,bbox_inches='tight'); fig.savefig(OPTIONAL_PANELS/f'{case}.pdf',bbox_inches='tight'); plt.close(fig)
pngs=sorted(OPTIONAL_PANELS.glob('CASE_*.png')); pdfs=sorted(OPTIONAL_PANELS.glob('CASE_*.pdf'))
if len(pngs)!=N_PILOT or len(pdfs)!=N_PILOT: raise RuntimeError(f'Expected {N_PILOT} PNG/PDF panels; found {len(pngs)}/{len(pdfs)}')
zip_path=Path(shutil.make_archive(str(FIGURES/'P1_E06_1_optional_expert_validation_panels'),'zip',root_dir=OPTIONAL_PANELS))
save_table(pd.DataFrame({'png':[str(p) for p in pngs],'pdf':[str(p) for p in pdfs]}),'09_optional_panel_export_inventory')
print(f'Optional panels verified: {len(pngs)} PNG + {len(pdfs)} PDF; ZIP={zip_path}')
display(Image(filename=str(pngs[0]),width=900))

## 9. Freeze the automated specification and create the P1-E07 hand-off

P1-E07 must read the JSON decision and locked CSV specification below. It will apply the frozen path to the complete recording cohort and perform automatic interval/epoch quality control. It must not retune preprocessing using diagnosis labels or downstream model performance.

In [ ]:
locked_spec=pd.DataFrame([
    {'parameter':'primary_path','value':selected},
    {'parameter':'bandpass_hz','value':'1-40'},
    {'parameter':'input_reference','value':'CAUEEG released AVG reference; no second CAR applied'},
    {'parameter':'target_sampling_rate_hz','value':FS_TARGET},
    {'parameter':'epoch_seconds','value':EPOCH_SECONDS},
    {'parameter':'epoch_overlap','value':'0%'},
    {'parameter':'target_epochs_per_recording','value':TARGET_EPOCHS},
    {'parameter':'minimum_epochs_per_recording','value':MIN_EPOCHS},
    {'parameter':'max_ica_components','value':MAX_ICA_COMPONENTS if selected=='enhanced' else 0},
    {'parameter':'max_interpolated_channels','value':MAX_REPAIRED_CHANNELS if selected=='enhanced' else 0},
    {'parameter':'path_selection_basis','value':'prespecified automatic quantitative QA only'},
    {'parameter':'specialist_review_required','value':False}])
locked_spec.to_csv(TABLES/'10_LOCKED_AUTOMATED_PREPROCESSING_SPECIFICATION.csv',index=False)

handoff=recordings[['recording_id','serial','edf_path','official_split','retained_seconds']].copy()
handoff['locked_path']=selected
handoff['automatic_qa_rule_version']='P1-E06.1-v1'
save_table(handoff,'11_P1_E07_FULL_COHORT_AUTOMATED_QA_HANDOFF')

spec_csv=TABLES/'10_LOCKED_AUTOMATED_PREPROCESSING_SPECIFICATION.csv'
handoff_csv=TABLES/'11_P1_E07_FULL_COHORT_AUTOMATED_QA_HANDOFF.csv'
decision_json=OUTPUT_ROOT/'p1_e06_1_automated_qa_preprocessing_path_lock.json'
run_record={
 'experiment':'P1-E06. automated quantitative signal-QA and preprocessing-path lock',
 'status':decision['status'],'selected_primary_path':selected,
 'input_p1_e05_1_root':str(E05_ROOT),'input_recordings':len(recordings),'input_intervals':len(intervals),
 'input_hashes':observed,'output_root':str(OUTPUT_ROOT),'seed':SEED,'n_pilot':N_PILOT,
 'pilot_source_split':'train','analysis_unit':'recording','participant_identity_verified':False,
 'diagnostic_labels_used':False,'manual_specialist_review_required':False,
 'expert_review_status':'future independent validation recommendation only',
 'classifier_performance_used':False,
 'decision_json':str(decision_json),'decision_sha256':sha256_file(decision_json),
 'locked_specification_csv':str(spec_csv),'locked_specification_sha256':sha256_file(spec_csv),
 'p1_e07_handoff_csv':str(handoff_csv),'p1_e07_handoff_sha256':sha256_file(handoff_csv),
 'python':sys.version,'platform':platform.platform(),'mne':mne.__version__}
(OUTPUT_ROOT/'run_record.json').write_text(json.dumps(run_record,indent=2),encoding='utf-8')
print(json.dumps(run_record,indent=2))
print('\nNEXT: P1-E07 full-cohort preprocessing and automated interval/epoch QA using the frozen',selected,'path.')